# Урок 4.5. Метаданные: основа безопасности и персонализации

Мы много раз говорили «фильтр применяется на этапе поиска». Пришло время
разобрать, **чем** фильтровать. Ответ — метаданными: тем, что мы записали
рядом с каждым чанком при индексации. Отдел, уровень секретности, арендатор,
версия, риск — всё это не украшение, а **несущая конструкция** и безопасности,
и персонализации RAG.

> **Идея урока:** в RAG права доступа, мультитенантность и версии — это не код
> вокруг поиска, а **условия внутри поискового запроса**. Кто что найдёт,
> решает `WHERE`, а не `if` после генерации.

Один и тот же вопрос «Какая вилка у сеньора?» должен дать HR цифры, а рядовому
сотруднику — вежливый отказ. Один и тот же индекс должен быть невидим для чужого
арендатора. Одна и та же политика отпусков должна отвечать актуальной редакцией,
а не прошлогодней. Всё это — работа метаданных.

## 🎯 Цели урока
- Спроектировать **схему метаданных** для корпуса и понять, зачем каждое поле.
- Реализовать **контроль доступа** как pre-filter (row-level security).
- Обеспечить **мультитенантность** — изоляцию арендаторов.
- Настроить **версионирование** документов (актуальная редакция, мягкое удаление).
- Увидеть, что метаданные дают не только защиту, но и **персонализацию**.


## Метаданные корпуса «Векторики»

Всё начинается с `data/manifest.json` — паспорта каждого документа. Это
единственный источник правды о том, кому документ виден и насколько он свежий.

In [ ]:
import sys
import os
import json
from pathlib import Path

MODULE_DIR = Path.cwd()
if not (MODULE_DIR / "rag_service").exists() and (MODULE_DIR.parent / "rag_service").exists():
    MODULE_DIR = MODULE_DIR.parent
sys.path.insert(0, str(MODULE_DIR))
os.environ.setdefault("RAG_JSON_LOGS", "0")

In [ ]:
from rag_service.config import DATA_DIR
from rag_service.store import KnowledgeBase
from rag_service.llm import OllamaLLM

kb = KnowledgeBase()
QDRANT_UP = kb.available() and kb.collection_exists()
OLLAMA_UP = OllamaLLM().available()
print(f"Qdrant: {'✓' if QDRANT_UP else '✗'} | Ollama: {'✓' if OLLAMA_UP else '✗'}")

In [ ]:
manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
print(f"{'doc_id':<20}{'отдел':<8}{'sens':<5}{'вер':<4}{'актуал':<8}{'источник':<11}владелец")
print("-" * 72)
for d in manifest["documents"]:
    print(f"{d['doc_id']:<20}{d['department']:<8}{d['sensitivity']:<5}"
          f"{d['version']:<4}{str(d['is_current']):<8}{d['source_type']:<11}{d['owner']}")

Пять полей, на которых держится весь урок:

- **`department`** — чей документ (`all` = общедоступный, `hr`, `eng`, ...).
- **`sensitivity`** — уровень секретности: 0 внутренний общий, 1 ограниченный,
  2 конфиденциальный.
- **`version` + `is_current`** — редакция и признак «действующая».
- **`source_type`** — `trusted` / `untrusted` (доверяем ли инструкциям внутри).
- **`owner`** — кто отвечает за документ (для аудита, не для фильтра).

При индексации (`ingest.py`) эти поля копируются в payload каждого чанка, плюс
добавляется вычисленный `risk_level` (урок 4.2). По ним и фильтруем.


## Кто спрашивает: Principal

Второй полюс — тот, кто задаёт вопрос. В `access.py` он описан как `Principal`:
кто пользователь, какого он арендатора, какие отделы ему видны, какой у него
уровень допуска (clearance), может ли он видеть ПДн. В реальном сервисе это
собирается из JWT/OIDC-токена; у нас — демо-токены.

In [ ]:
from rag_service.access import DEMO_PRINCIPALS

for token, p in DEMO_PRINCIPALS.items():
    print(f"{token}")
    print(f"    tenant={p.tenant_id}, departments={p.departments}, "
          f"clearance={p.clearance}, may_see_pii={p.may_see_pii}")

## Принцип 1. Контроль доступа = фильтр поиска (row-level security)

Правило доступа `can_read` простое и читается как строка SQL WHERE:

```
видно, если:  tenant совпал  И  sensitivity <= clearance  И  (отдел мой ИЛИ отдел = all)
```

`build_filter(principal)` превращает это в фильтр Qdrant. Посмотрим, как он
выглядит для разных ролей.

In [ ]:
from rag_service.access import build_filter, describe_filter

labels = {"demo-employee-token": "сотрудник",
          "demo-engineer-token": "инженер",
          "demo-hr-token": "HR",
          "demo-other-tenant-token": "чужой tenant (acme)"}

for token, label in labels.items():
    flt = build_filter(DEMO_PRINCIPALS[token])
    print(f"{label:22s}: {describe_filter(flt)}")

Теперь — живая проверка на инженерных документах. `guide-deploy` и
`inc-2026-014` помечены `department=eng`. Рядовой сотрудник (`departments=('all',)`)
их видеть не должен, инженер (`departments=('all','eng')`) — должен.

In [ ]:
from rag_service.embeddings import get_embedder
from rag_service.retrieval import LexicalIndex, retrieve
from rag_service.observability import configure_logging

configure_logging("ERROR", json_logs=False)
if QDRANT_UP:
    emb = get_embedder()
    lex = LexicalIndex.from_kb(kb)
    for q in ["Когда нельзя деплоить в прод?", "Что было в инциденте с биллингом?"]:
        print(f"\nВопрос: {q}")
        for token in ("demo-employee-token", "demo-engineer-token"):
            r = retrieve(q, DEMO_PRINCIPALS[token], kb, emb, lexical=lex)
            docs = sorted({h.doc_id for h in r.hits})
            print(f"  {labels[token]:10s}: {docs}")
else:
    print("Мой прогон: 'Когда нельзя деплоить' — у сотрудника guide-deploy НЕ найден,")
    print("у инженера найден. Инцидент inc-2026-014 (dept=eng, sens=1) — тот же расклад.")

В моём прогоне `guide-deploy` и `inc-2026-014` появлялись **только** у
инженера. У рядового сотрудника их нет в выдаче вообще — не «спрятаны в ответе»,
а не найдены. Это ключевая разница.

### Почему pre-filter, а не post-filter (ещё раз, но глубже)

Соблазн новичка: найти всё, а потом в ответе показать только разрешённое. Три
причины, почему так делать **нельзя**:

1. **Утечка через модель.** Если секретный чанк попал в промпт, модель включит
   его в ответ — просьба «не показывай» ненадёжна (это мы видели в уроке 4.2).
2. **Утечка через метаданные.** Даже сам факт «документ существует» бывает
   секретом (например, `inc-2026-014` — что инцидент вообще был).
3. **Утечка через тайминги и логи.** Найденный, но «скрытый» чанк всё равно
   прошёл через пайплайн, попал в логи, повлиял на латентность.

Единственный корректный момент — **до поиска**: фильтр уходит прямо в запрос к
индексу. Qdrant применяет его во время обхода HNSW (filtered search из урока
2.4), поэтому недоступный документ не просто прячется — он не участвует в поиске.


## Принцип 2. Мультитенантность: стена между арендаторами

Если ваш RAG обслуживает несколько компаний (или команд, изолированных по
данным) — это **мультитенантный** сервис, и первое правило B2B SaaS: арендатор
**никогда** не видит данные другого. Один ошибочный запрос без `tenant_id` в
фильтре — и это утечка на первую полосу.

У нас есть `demo-other-tenant-token` — пользователь арендатора `acme` с
**максимальным** уровнем допуска (clearance=2) и доступом к eng. Проверим, что
даже он не увидит ни одного документа «Векторики».

In [ ]:
if QDRANT_UP:
    acme = DEMO_PRINCIPALS["demo-other-tenant-token"]
    r = retrieve("Сколько дней отпуска? VPN? зарплата?", acme, kb, emb, lexical=lex)
    print(f"acme (clearance=2, но tenant=acme) нашёл документов: {len(r.hits)}")
    print("Ожидаем 0: изоляция арендатора сильнее любого уровня допуска.")
else:
    print("Мой прогон: acme при clearance=2 нашёл 0 документов «Векторики».")
    print("tenant_id в фильтре — первое и несокрушимое условие.")

Ноль документов — при максимальном допуске. `tenant_id` в фильтре стоит
первым и не обходится уровнем доступа: допуск разграничивает **внутри**
арендатора, а `tenant_id` разделяет арендаторов между собой.

### Две архитектуры мультитенантности

| Подход | Как | Когда |
|---|---|---|
| **Shared collection + фильтр** (наш) | все арендаторы в одной коллекции, `tenant_id` в каждом фильтре | много мелких арендаторов, экономия ресурсов |
| **Collection-per-tenant** | своя коллекция на арендатора | крупные арендаторы, жёсткие требования изоляции/комплаенса |

Наш подход дешевле и проще, но требует **дисциплины**: забыл `tenant_id` в одном
фильтре — и стены нет. Поэтому фильтр собирается **централизованно** в
`build_filter`, а не руками в каждом эндпоинте. Collection-per-tenant надёжнее
физически (невозможно случайно смешать), но дороже в эксплуатации.


## Принцип 3. Версионирование: отвечать актуальным

Антипаттерн №1 из урока 4.1 — устаревшие данные. Лечится он метаданными
`version` + `is_current`. В корпусе две редакции отпуска; фильтр `is_current`
(параметр `only_current`, по умолчанию True) отбирает только действующую.

In [ ]:
if QDRANT_UP:
    emp = DEMO_PRINCIPALS["demo-employee-token"]
    for only_current in (True, False):
        r = retrieve("Сколько дней отпуска положено?", emp, kb, emb,
                     lexical=lex, only_current=only_current)
        versions = sorted({(h.doc_id, h.metadata.get("version"))
                           for h in r.hits if h.doc_id == "policy-vacation"})
        print(f"only_current={only_current}: версии policy-vacation в выдаче = {versions}")
    print("\nПо умолчанию only_current=True → в выдаче только действующая редакция (v2).")
else:
    print("Мой прогон: only_current=True → только v2 (2026); False → обе (v1 2023 + v2 2026).")


### Как жить с версиями

- **`is_current`** — самый частый фильтр: «отвечай только действующим».
  Пользователю не нужна прошлогодняя редакция.
- **Мягкое удаление, а не физическое.** Старую версию помечаем `is_current=False`,
  но **храним**: она нужна для аудита («а что было в правилах в марте?») и для
  вопросов «когда изменилось». Наш `set_payload`-подход (урок 2.4) меняет флаг
  без пересчёта вектора.
- **Детерминированные id.** id чанка = `tenant:doc_id:vN:номер`. Новая версия —
  это новый `vN`, она не перезаписывает старую вслепую; переиндексация одной
  версии идемпотентна (тот же id перезаписывает тот же чанк).
- **Version pinning.** Иногда нужно спросить конкретную версию («что было в
  редакции 2023?») — это фильтр `version=1, only_current=False` (упражнение 2).


## Принцип 4. Риск как метаданное (и почему фильтр должен быть везде)

Уровень риска (`risk_level`) из урока 4.2 — тоже метаданное, и фильтруется тем
же механизмом. Но здесь важная тонкость, которую легко упустить: у нас **два**
пути поиска — плотный (Qdrant-фильтр) и BM25 (в памяти). Фильтр надо применять
на **обоих**, иначе лексический путь станет дырой в обороне.

In [ ]:
if QDRANT_UP:
    emp = DEMO_PRINCIPALS["demo-employee-token"]
    for max_risk in ("medium", "high"):
        r = retrieve("универсальный пароль VPN", emp, kb, emb, lexical=lex, max_risk=max_risk)
        poison = any(h.doc_id == "faq-vpn-community" for h in r.hits)
        print(f"max_risk={max_risk:7s}: отравленный faq-vpn-community в выдаче = {poison}")
    print("\nmedium (дефолт) отсекает risk_level=2. Фильтр риска применяется И в dense, И в BM25.")
else:
    print("Мой прогон: max_risk=medium → faq отсечён (False); max_risk=high → пропущен (True).")
    print("Тонкость: BM25 ищет в памяти по всей коллекции, поэтому risk_level надо")
    print("проверять и в его предикате allow — иначе отравленный чанк просочится мимо dense-фильтра.")


> Это не выдуманный пример: при подготовке урока `faq-vpn-community` (risk_level=2)
> действительно просачивался через BM25 при `max_risk=medium`, потому что
> лексический путь не проверял риск. Мораль общая для любого метаданного-фильтра:
> **фильтруйте на всех путях поиска одинаково.** Гибридный поиск (урок 4.4)
> удобен, но удваивает число мест, где легко забыть про безопасность.


## Метаданные — это не только про запрет

До сих пор мы запрещали. Но те же метаданные дают **персонализацию**: один
вопрос — разные ответы по роли, и это фича, а не только защита. Спросим про
запрет деплоя от лица сотрудника и инженера на живой модели.

In [ ]:
from rag_service.pipeline import RagPipeline

if QDRANT_UP and OLLAMA_UP:
    pipe = RagPipeline(kb=kb, llm=OllamaLLM())
    pipe.build_lexical_index()
    q = "Когда нельзя деплоить в прод?"
    for token in ("demo-employee-token", "demo-engineer-token"):
        res = pipe.ask(q, DEMO_PRINCIPALS[token])
        print(f"\n{labels[token]} → outcome={res.outcome}")
        print(f"  источники: {[s['doc_id'] for s in res.sources]}")
        print(f"  ответ: {res.answer[:200]}")
else:
    print("Мой прогон: сотрудник → отказ/нет источника (guide-deploy для него невидим),")
    print("инженер → ответ по guide-deploy (пятница после 15:00, code freeze).")
    print("Один вопрос, один индекс, один промпт — разный ответ по роли. Это персонализация.")

Ключевое — в **источниках**: у инженера в них есть `guide-deploy`, у
сотрудника его нет вовсе (deploy — документ отдела eng). Поэтому инженер получает
точный ответ (пятница после 15:00, новогодний code freeze), а сотрудник — в
лучшем случае расплывчатый ответ без опоры на нужный документ: ему просто нечем
ответить, потому что документ для него не найден. Один вопрос, один индекс, один
промпт — **разный результат по роли**. Так метаданные превращаются из решётки
запретов в инструмент персонализации: показать пользователю ровно его срез
знаний. Тот же механизм — основа A/B редакций, региональных версий, языковых
вариантов документа.

> Напоминание из урока 3.3: единичные ответы 3B-модели на GPU нестабильны —
> сотруднику может выпасть и честный отказ, и уклончивый ответ. Устойчив здесь
> именно список источников: `guide-deploy` доступен только инженеру.


## Как это работает в Qdrant (recap урока 2.4)

Чтобы фильтр не превращался в полный перебор коллекции, поля, по которым мы
фильтруем, должны иметь **payload-индексы**. Наш `KnowledgeBase.ensure_collection`
создаёт их для всех полей фильтра.

In [ ]:
if QDRANT_UP:
    print("Поля с payload-индексами (по ним фильтр работает быстро):")
    for field, schema in KnowledgeBase.INDEXED_FIELDS.items():
        print(f"  {field:<14} {schema}")
    print("\nQdrant применяет эти условия ВО ВРЕМЯ обхода HNSW (filtered search):")
    print("недоступные точки не просто прячутся из выдачи — они не посещаются.")
else:
    print("Индексируемые поля:", list(KnowledgeBase.INDEXED_FIELDS))

Без индекса фильтр по `tenant_id` на большой коллекции — это линейное
сканирование (как `WHERE` по колонке без индекса в SQL). С индексом Qdrant
совмещает фильтрацию с обходом графа HNSW: релевантные И разрешённые точки
находятся за один проход, а выдача не «дырявится», как при пост-фильтрации
(проклятие post-фильтра из урока 2.2).


## Чек-лист: проектируем схему метаданных

Прежде чем индексировать корпус, ответьте на вопросы — каждый рождает поле:

- **Кто это видит?** → `department`, `sensitivity`, роли/группы.
- **Чей это арендатор?** → `tenant_id` (даже если сейчас один — заложите сразу).
- **Актуально ли?** → `version`, `is_current`, `updated_at`.
- **Доверяем ли источнику?** → `source_type`, вычисляемый `risk_level`.
- **Кто отвечает?** → `owner` (для аудита).
- **Нужны ли срезы?** → язык, регион, продукт — всё, по чему захотите
  персонализировать или фильтровать.

Правила гигиены:
- поля фильтра — **простые скаляры** (str/int/bool), с payload-индексами;
- фильтр собирается **централизованно** (`build_filter`), не руками в эндпоинтах;
- метаданные пишутся при **индексации** и версионируются вместе с чанком;
- всё, по чему фильтруете, проверяется на **всех** путях поиска (dense и BM25).


## ⚠️ Частые ошибки

- **Пост-фильтрация вместо pre-filter.** Нашли всё, скрыли лишнее в ответе — это
  утечка (через модель, через факт существования, через логи). Фильтр — в запрос.
- **Забыли `tenant_id`.** Одна строка без арендатора в фильтре — межтенантная
  утечка. Собирайте фильтр централизованно, не копируйте условия руками.
- **Фильтр только на одном пути.** Гибридный поиск = два пути (dense + BM25).
  Метаданное-фильтр должен стоять на обоих (наш баг с BM25 и risk_level).
- **Физическое удаление старых версий.** Теряете аудит и историю. Мягкое удаление
  (`is_current=False`), храните версии.
- **Нет payload-индексов.** Фильтр по неиндексированному полю на большой
  коллекции — полное сканирование. Индексируйте все поля фильтра.
- **Права в промпте, а не в поиске.** «Модель, не показывай зарплаты чужим» —
  это не контроль доступа, это пожелание. Контроль — `WHERE` в индексе.
- **Списки в метаданных.** У большинства хранилищ значение метаданного — скаляр.
  Права кодируйте числом (`clearance`/`sensitivity`) и `department` + `MatchAny`,
  а не списком ролей в одном поле.


## Упражнения

1. **Матрица доступа.** Постройте таблицу: строки — 4 демо-роли, столбцы — все
   `doc_id` корпуса, в клетке ✓/✗ по `can_read`. Сверьте с живым поиском: для
   каждой роли выведите множество найденных `doc_id` по широкому запросу и
   убедитесь, что оно совпадает с ✓-клетками. Это ваш регрессионный тест прав.

2. **Version pinning.** Реализуйте вопрос к конкретной редакции: «Сколько дней
   отпуска было по правилам 2023 года?» Соберите фильтр с `version=1,
   only_current=False` (подсказка: `build_filter(..., extra_must=[...])`) и
   убедитесь, что находится v1 с 28 днями, а не v2 с 31.

3. **Новый арендатор и новый отдел.** Добавьте в `DEMO_PRINCIPALS` роль
   `demo-finance-token` (departments=('all','finance'), clearance=1) и один
   документ отдела finance в manifest. Проиндексируйте и проверьте: finance
   видит свой документ, а инженер и сотрудник — нет. Это полный цикл добавления
   среза данных.


## Мини-квиз

<details><summary><b>1. Почему контроль доступа в RAG — это фильтр поиска, а не проверка в ответе?</b></summary>

Потому что если недоступный чанк попал в промпт, он уже утёк: модель включит его
в ответ (просьба «не показывай» ненадёжна), сам факт существования документа
может быть секретом, и чанк уже засветился в логах и латентности. Единственный
корректный момент — до поиска: фильтр уходит в запрос к индексу, и недоступная
точка не участвует в поиске вообще (filtered HNSW).
</details>

<details><summary><b>2. Пользователь arendatора acme имеет clearance=2. Почему он не видит документы «Векторики»?</b></summary>

Потому что `tenant_id` в фильтре стоит первым и не обходится уровнем допуска.
Clearance разграничивает доступ **внутри** одного арендатора (кто из «Векторики»
видит конфиденциальное), а `tenant_id` разделяет арендаторов **между** собой.
Максимальный допуск в чужом арендаторе = ноль доступа к данным другого.
</details>

<details><summary><b>3. Зачем хранить старые версии документов, а не удалять их?</b></summary>

Для аудита и истории: «какие были правила в марте?», «когда изменилась сумма
отпускных?». Физическое удаление стирает эту возможность. Правильно — мягкое
удаление: пометить `is_current=False` (по умолчанию такие в выдачу не попадают),
но чанки сохранить. При этом детерминированные id гарантируют, что новая версия
не смешается со старой.
</details>

<details><summary><b>4. Что опасного в гибридном поиске с точки зрения контроля доступа?</b></summary>

У гибрида два пути: плотный (фильтр уходит в Qdrant) и BM25 (поиск в памяти по
всей коллекции с предикатом allow). Метаданное-фильтр надо применять на **обоих**
одинаково — иначе лексический путь пропустит то, что плотный отсёк. У нас именно
так `risk_level` какое-то время не проверялся в BM25, и отравленный чанк
просачивался. Каждый новый путь поиска удваивает места, где легко забыть фильтр.
</details>

<details><summary><b>5. Почему поля-фильтры должны быть простыми скалярами с payload-индексами?</b></summary>

Скаляры (str/int/bool) — потому что большинство векторных хранилищ не поддерживают
списки в значениях метаданных; права кодируют числом (clearance/sensitivity) и
`department` через MatchAny, а не списком ролей в одном поле. Payload-индексы —
потому что без индекса фильтр по полю на большой коллекции превращается в полное
сканирование; с индексом Qdrant фильтрует прямо во время обхода HNSW.
</details>


## Решения

Три упражнения — три операции над метаданными: проверить права таблицей,
спросить конкретную редакцию и добавить новый срез данных с нуля.

**Решение 1 — матрица доступа.** `can_read` — чистая функция (тенант + отдел +
допуск), её легко прогнать по всем парам (роль, документ) и получить таблицу
прав. А потом — сверить с живым поиском: множество найденных `doc_id` должно
совпасть с ✓-клетками. Это регрессионный тест: сломается правило доступа —
разойдётся таблица.

In [ ]:
from rag_service.access import DEMO_PRINCIPALS, build_filter
from rag_service.ingest import load_manifest

roles = {"сотрудник": "demo-employee-token", "инженер": "demo-engineer-token",
         "HR": "demo-hr-token", "acme": "demo-other-tenant-token"}
doc_meta = {d["doc_id"]: (d.get("department", "all"), int(d.get("sensitivity", 0)))
            for d in load_manifest()}

print("матрица прав (can_read):")
print(f"{'doc_id':22s}" + "".join(f"{r[:9]:>10s}" for r in roles))
for doc_id, (dept, sens) in doc_meta.items():
    cells = []
    for token in roles.values():
        p = DEMO_PRINCIPALS[token]
        cells.append("✓" if p.can_read(department=dept, sensitivity=sens,
                                        tenant_id="vectorika") else "✗")
    print(f"{doc_id:22s}" + "".join(f"{c:>10s}" for c in cells))

if QDRANT_UP:
    print("\nсверка с живым поиском (широкий невод, only_current=False, max_risk=high):")
    for role, token in roles.items():
        p = DEMO_PRINCIPALS[token]
        r = retrieve("отпуск больничный vpn зарплата деплой инцидент оборудование",
                     p, kb, emb, lexical=lex, candidates_k=100, final_k=100,
                     min_similarity=0.0, only_current=False, max_risk="high", use_rerank=False)
        found = {h.doc_id for h in r.hits}
        allowed = {d for d, (dp, sn) in doc_meta.items()
                   if p.can_read(department=dp, sensitivity=sn, tenant_id="vectorika")}
        print(f"  {role:10s}: найдено==разрешено -> {found == allowed}  ({len(found)} док.)")

Матрица читается сразу: `guide-deploy` и `inc-2026-014` (отдел `eng`) видны
только инженеру; `hr-salary-bands` — только HR; `policy-security` (sensitivity=1)
скрыт от рядового сотрудника (clearance=0); а `acme` не видит **ничего** —
граница арендатора сильнее любого допуска. И живой поиск совпал с таблицей для
всех ролей.

Тонкость сверки: `can_read` проверяет только тенант/отдел/допуск, а боевой
`build_filter` добавляет ещё `is_current` и `risk_level`. Чтобы сравнивать
одно с одним, в живом поиске мы сняли эти два условия (`only_current=False`,
`max_risk="high"`). В проде они, наоборот, включены — и `faq-vpn-community`
(risk_level=2) в обычной выдаче отсеклётся, хотя по правам он доступен всем.

**Решение 2 — version pinning.** Иногда нужен ответ по конкретной редакции
(«а как было в 2023?»). `build_filter` принимает `extra_must`, куда кладём
условие `version=1`, и снимаем `only_current` (старая редакция помечена
`is_current=False`).

In [ ]:
if QDRANT_UP:
    import re
    from qdrant_client import models

    query = emb.encode_query("сколько дней отпуска положено")
    emp = DEMO_PRINCIPALS["demo-employee-token"]

    flt_v1 = build_filter(emp, only_current=False, extra_must=[
        models.FieldCondition(key="version", match=models.MatchValue(value=1))])
    flt_current = build_filter(emp)   # дефолт: only_current=True -> действующая v2

    for label, flt in [("pinned version=1 (2023)", flt_v1),
                       ("по умолчанию (действующая)", flt_current)]:
        hits = kb.search(query, k=5, flt=flt)
        vac = next((h for h in hits if h.doc_id == "policy-vacation"), None)
        if vac:
            # берём число ИМЕННО перед «календарн», а не первое в тексте:
            # у v2 рядом стоят и «31 календарный день», и «28 дней по ТК»
            m = re.search(r"(\d+)\s+календарн", vac.text)
            days = m.group(1) if m else "?"
            print(f"{label:32s}: policy-vacation v{vac.metadata.get('version')} "
                  f"-> дней отпуска: {days}")
else:
    print("Нужен Qdrant. version=1 через extra_must + only_current=False -> v1 (28 дней),")
    print("дефолт only_current=True -> v2 (31 день).")

Пиннинг сработал: с `version=1` находится редакция 2023 года (28 дней), с
дефолтным фильтром — действующая 2026 (31 день). Это тот же механизм, что
страхует нас от антипаттерна №1 (устаревшие данные), только повёрнутый в другую
сторону: обычно `is_current` **прячет** старьё, а здесь мы осознанно просим
именно его. Ключевое — это по-прежнему фильтр поиска, а не пост-обработка:
если бы v1 просто «отфильтровали в ответе», модель всё равно увидела бы обе
редакции в контексте.

**Решение 3 — новый арендатор и новый отдел.** Полный цикл добавления среза
данных: новая роль + новый документ + индексация + проверка изоляции. Соберём
`Principal` отдела finance и документ с `department=finance`, проиндексируем его
в рабочую коллекцию и проверим, кто его видит. В конце — уборка.

In [ ]:
if QDRANT_UP:
    from rag_service.access import Principal
    from rag_service.ingest import build_chunks

    finance = Principal(user_id="u-4004", departments=("all", "finance"),
                        clearance=1, roles=("employee", "finance"))
    fin_doc = {"doc_id": "policy-expenses", "title": "Возмещение расходов (finance)",
               "file": "-", "version": 1, "source_type": "trusted",
               "department": "finance", "sensitivity": 1, "is_current": True}
    fin_raw = ("Возмещение расходов сотрудникам компании «Векторика».\n\n"
               "Командировочные согласует руководитель отдела финансов. "
               "Лимит суточных — 3500 рублей, чеки прикладываются в течение 5 дней.")

    fin_chunks = build_chunks(fin_doc, fin_raw, sanitize=True)      # рубеж 0 из 4.2
    kb.upsert(fin_chunks, emb.encode_documents([c.text for c in fin_chunks]))
    try:
        checks = [("finance (новая роль)", finance),
                  ("инженер", DEMO_PRINCIPALS["demo-engineer-token"]),
                  ("сотрудник", DEMO_PRINCIPALS["demo-employee-token"])]
        for role, p in checks:
            seen = kb.fetch_all(flt=build_filter(p, only_current=False, max_risk="high"))
            has_fin = any(h.doc_id == "policy-expenses" for h in seen)
            print(f"  {role:22s}: видит policy-expenses = {has_fin}")
    finally:
        kb.delete_document("policy-expenses")     # уборка: срез был временный
        print("  (уборка: policy-expenses удалён из коллекции)")
else:
    print("Нужен Qdrant. finance видит свой документ, инженер и сотрудник — нет.")

Изоляция сработала на первом же прогоне: `finance` видит свой документ,
инженер и рядовой сотрудник — нет, потому что `department=finance` не входит в их
`departments`. Обратите внимание, что делать пришлось ровно две вещи — **новую
роль** (`Principal` с нужными `departments`/`clearance`) и **новый документ с
метаданными** — а весь механизм доступа (`build_filter`) не изменился ни на
строчку. В этом и мораль модуля: права, тенанты, версии и отделы — это не код в
обработчике, а метаданные плюс один фильтр поиска. Добавить срез данных — значит
дописать метаданные, а не переписать сервис.

В проде `Principal` пришёл бы из JWT/OIDC (claim с отделом), а документ — из
`manifest.json` с полной переиндексацией; здесь мы срезали углы (upsert одного
документа и уборка), чтобы показать именно контроль доступа, не гоняя весь ETL.

## Итоги

- Метаданные — **несущая конструкция** RAG: права, тенанты, версии, риск — всё
  это условия внутри поискового запроса, а не код вокруг него.
- **Контроль доступа = pre-filter** (row-level security): недоступный документ
  не участвует в поиске, а не прячется в ответе.
- **Мультитенантность**: `tenant_id` первым в фильтре, сильнее любого допуска;
  shared+фильтр (дёшево, нужна дисциплина) или collection-per-tenant (надёжнее).
- **Версионирование**: `is_current` по умолчанию, мягкое удаление, детерминированные
  id — лекарство от антипаттерна устаревших данных.
- Фильтр применяйте на **всех** путях поиска (dense и BM25) — гибрид удваивает
  места, где можно забыть про безопасность.
- Те же метаданные дают **персонализацию**: один вопрос — разный ответ по роли.

## Что дальше

Это был последний содержательный урок модуля. Впереди — **4.6 «Закрепление»**:
сводный квиз по всему продвинутому RAG и практика — собрать изученное в единую
картину. После него модуль 4 закрыт, и курс идёт к локальным LLM (модуль 5).

## Полезные ссылки
- Qdrant — фильтрация и payload-индексы: <https://qdrant.tech/documentation/concepts/filtering/>
- Multitenancy в Qdrant: <https://qdrant.tech/documentation/guides/multiple-partitions/>
- Row-level security (концепция, PostgreSQL): <https://www.postgresql.org/docs/current/ddl-rowsecurity.html>
- OWASP LLM06 (Sensitive Information Disclosure): <https://genai.owasp.org/>
